In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv("../data/fake reviews dataset.csv")

In [4]:
data.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [6]:
data['label'].value_counts()

label
CG    20216
OR    20216
Name: count, dtype: int64

In [7]:
data['label'] = data['label'].map({'CG':0, 'OR':1})

In [8]:
data = data[['text_', 'label']]

In [9]:
data.head()

,text_,label
0,"Love this! Well made, sturdy, and very comfor...",0
1,"love it, a great upgrade from the original. I...",0
2,This pillow saved my back. I love the look and...,0
3,"Missing information on how to use it, but it i...",0
4,Very nice set. Good quality. We have had the s...,0


In [10]:
import nltk
import string
from nltk.corpus import stopwords

In [11]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
def clean_text(text):
    
    # convert text to lowercase
    text = text.lower()
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # split words
    words = text.split()
    
    # remove stopwords
    words = [word for word in words if word not in stopwords.words('english')]
    
    # join words again
    return " ".join(words)

In [15]:
data['clean_review'] = data['text_'].apply(clean_text)

In [16]:
data.head()

,text_,label,clean_review
0,"Love this! Well made, sturdy, and very comfor...",0,love well made sturdy comfortable love itvery ...
1,"love it, a great upgrade from the original. I...",0,love great upgrade original ive mine couple years
2,This pillow saved my back. I love the look and...,0,pillow saved back love look feel pillow
3,"Missing information on how to use it, but it i...",0,missing information use great product price
4,Very nice set. Good quality. We have had the s...,0,nice set good quality set two months


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(data['clean_review'])

y = data['label']

X.shape


(40432, 5000)

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [22]:
print(X_train.shape)
print(X_test.shape)

(32345, 5000)
(8087, 5000)


In [23]:
from sklearn.linear_model import LogisticRegression

In [24]:
model = LogisticRegression()

In [25]:
model.fit(X_train, y_train)

LogisticRegression()

In [26]:
predictions = model.predict(X_test)

In [27]:
from sklearn.metrics import accuracy_score

In [28]:
accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.8804253740571287


In [29]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

[[3488  528]
 [ 439 3632]]


In [30]:
review = ["This product is absolutely amazing and best purchase ever"]

In [31]:
review_vector = vectorizer.transform(review)

In [32]:
prediction = model.predict(review_vector)

print(prediction)

[1]


In [33]:
if prediction[0] == 0:
    print("Fake Review ❌")
else:
    print("Genuine Review ✅")

Genuine Review ✅


In [34]:
import pickle

pickle.dump(model, open("../model/fake_review_model.pkl", "wb"))
pickle.dump(vectorizer, open("../model/vectorizer.pkl", "wb"))